# 1. Importing Data

In [ ]:
import pandas as pd
import requests
from io import BytesIO

url = 'https://1drv.ms/x/c/0a8e8c1de260ebbd/IQBpBir-_RxXR6SXigNyBSO_AUOl_SQUEs6Lmz3Jm-hle5E?e=qyVQ0v&download=1'
# AI told me the trick to add a "&download=1" manually at the end of the url
# this makes the requesting module work

response = requests.get(url)

with open("data/output_excel.xlsx", "wb") as data:
    data.write(response.content)

In [145]:
loaded_data = {}    # create an empty dictionary to collect data

with open("data/output_excel.xlsx", "rb") as data:
    for i in range(0, 8):
        loaded_data[f"sheet_{i}"] = pd.read_excel(data, sheet_name=i, engine='openpyxl') 
        # filled up the dictionary with data as pandas dataframes
        # labelling each sheet by index 0, 1, 2, 3, 4, ,5, 6, 7

for keys in loaded_data:
    display(loaded_data[keys])    # inspecting what each sheet is for

,"Penn World Table, version 10.0"
0,NaN
1,"This file contains the data of PWT 10.0, as av..."
2,Please refer to www.ggdc.net/pwt for extensive...
3,NaN
4,"When using these data, please refer to the fol..."
5,"Feenstra, Robert C., Robert Inklaar and Marcel..."
6,NaN
7,Note
8,Revision of June 2021. Please consult the chan...


,Variable name,Variable definition
0,Identifier variables,NaN
1,countrycode,3-letter ISO country code
2,country,Country name
3,currency_unit,Currency unit
4,year,Year
...,...,...
62,pl_g,"Price level of government consumption, price ..."
63,pl_x,"Price level of exports, price level of USA GDP..."
64,pl_m,"Price level of imports, price level of USA GDP..."
65,pl_n,"Price level of the capital stock, price level ..."


,countrycode,country,currency_unit,year,rgdpna
0,AGO,Angola,Kwanza,1970,54237.054688
1,AGO,Angola,Kwanza,1971,57491.277344
2,AGO,Angola,Kwanza,1972,57606.261719
3,AGO,Angola,Kwanza,1973,62272.367188
4,AGO,Angola,Kwanza,1974,64202.808594
...,...,...,...,...,...
7745,ZWE,Zimbabwe,US Dollar,2015,42008.199219
7746,ZWE,Zimbabwe,US Dollar,2016,42325.726562
7747,ZWE,Zimbabwe,US Dollar,2017,44316.742188
7748,ZWE,Zimbabwe,US Dollar,2018,46457.097656


,countrycode,year,rnna
0,AGO,1970,295517.625000
1,AGO,1971,314195.093750
2,AGO,1972,332435.843750
3,AGO,1973,352647.906250
4,AGO,1974,373267.718750
...,...,...,...
7715,ZWE,2015,64916.476562
7716,ZWE,2016,66257.859375
7717,ZWE,2017,67627.562500
7718,ZWE,2018,69059.625000


,countrycode,year,emp
0,AGO,1970,3.666207
1,AGO,1971,3.742484
2,AGO,1972,3.853271
3,AGO,1973,3.987807
4,AGO,1974,4.130696
...,...,...,...
7715,ZWE,2015,6.393752
7716,ZWE,2016,6.504374
7717,ZWE,2017,6.611773
7718,ZWE,2018,6.714952


,countrycode,year,hc
0,AGO,1970,1.015686
1,AGO,1971,1.018196
2,AGO,1972,1.020712
3,AGO,1973,1.023234
4,AGO,1974,1.025762
...,...,...,...
7715,ZWE,2015,2.584653
7716,ZWE,2016,2.616257
7717,ZWE,2017,2.648248
7718,ZWE,2018,2.680630


,countrycode,year,pop
0,AGO,1970,5.890365
1,AGO,1971,6.040777
2,AGO,1972,6.248552
3,AGO,1973,6.496962
4,AGO,1974,6.761380
...,...,...,...
7715,ZWE,2015,13.814629
7716,ZWE,2016,14.030331
7717,ZWE,2017,14.236595
7718,ZWE,2018,14.438802


,countrycode,year,labsh
0,AGO,1970,0.284385
1,AGO,1971,0.284385
2,AGO,1972,0.284385
3,AGO,1973,0.284385
4,AGO,1974,0.284385
...,...,...,...
7715,ZWE,2015,0.533381
7716,ZWE,2016,0.533381
7717,ZWE,2017,0.533381
7718,ZWE,2018,0.533381


# 2. Cleansing Data

## 2.1 Removing redundant variables

In [146]:
# slightly confused by what "labsh" means. just curious.
df = loaded_data["sheet_1"]

display(df[df["Variable name"] == "labsh"])

,Variable name,Variable definition
32,labsh,Share of labour compensation in GDP at current...


In [147]:
# don't really need the sheets for introduction, population, and labour share (i.e., sheet 0, 6, 7)
loaded_data = loaded_data
for i in range(0, 8):
    if ((i == 0) or (i == 6) or (i == 7)):
        del loaded_data[f"sheet_{i}"]

loaded_data.keys()

dict_keys(['sheet_1', 'sheet_2', 'sheet_3', 'sheet_4', 'sheet_5'])

In [148]:
for keys in loaded_data:
    display(loaded_data[f"{keys}"].head(0))

#preserving sheet 1, 2, 3, 4, 5

,Variable name,Variable definition


,countrycode,country,currency_unit,year,rgdpna


,countrycode,year,rnna


,countrycode,year,emp


,countrycode,year,hc


In [149]:
# view the definitions of variables
df = loaded_data["sheet_1"]

for variable in ["rgdpna", "rnna", "emp", "hc"]:
    display(df[df["Variable name"] == variable])

,Variable name,Variable definition
25,rgdpna,Real GDP at constant 2017 national prices (in ...


,Variable name,Variable definition
28,rnna,Capital stock at constant 2017 national prices...


,Variable name,Variable definition
10,emp,Number of persons engaged (in millions)


,Variable name,Variable definition
12,hc,"Human capital index, based on years of schooli..."


## 2.2 Removing empty values

In [ ]:
sheet_2 = loaded_data['sheet_2']
print(len(sheet_2))
empty_emp = sheet_2[sheet_2.isnull().any(axis=1)]
display(empty_emp)
print(len(empty_emp))

dav = sheet_2[sheet_2['countrycode'] == "DAV"]
display(dav)
print(len(dav))



7750


,countrycode,country,currency_unit,year,rgdpna
2587,FRA,France,Euro,2020,NaN
2588,FRA,France,Euro,2021,NaN
2589,FRA,France,Euro,2022,NaN
2590,FRA,France,Euro,2023,NaN
2591,FRA,France,Euro,2024,NaN
3307,IRL,Ireland,Euro,1945,NaN
3308,IRL,Ireland,Euro,1946,NaN
3309,IRL,Ireland,Euro,1947,NaN
3310,IRL,Ireland,Euro,1948,NaN
3311,IRL,Ireland,Euro,1949,NaN


13


,countrycode,country,currency_unit,year,rgdpna
1918,DAV,Davidland,Dave Bucks,2000,297615.966975
1919,DAV,Davidland,Dave Bucks,2001,406081.830039
1920,DAV,Davidland,Dave Bucks,2002,280928.632931
1921,DAV,Davidland,Dave Bucks,2003,414515.462764
1922,DAV,Davidland,Dave Bucks,2004,318230.406660
1923,DAV,Davidland,Dave Bucks,2005,393447.098524
1924,DAV,Davidland,Dave Bucks,2006,365408.899996
1925,DAV,Davidland,Dave Bucks,2007,290036.306425
1926,DAV,Davidland,Dave Bucks,2008,380500.602547
1927,DAV,Davidland,Dave Bucks,2009,302530.516234


20


In [151]:
# the guide's method doesn't work here, and I didn't work out a really viable method
# these tricks are taught by claude:

#get all countries in sheet 2
countries_rgdpna = set(loaded_data['sheet_2']['countrycode'])
# Get all countries appearing in sheet 4
countries_emp = set(loaded_data['sheet_4']['countrycode'])
# Find countries in rgdpna but absent from all other sheets
missing = countries_rgdpna - countries_emp
display(missing)

{'DAV'}

In [152]:
loaded_data['sheet_2'] = loaded_data['sheet_2'][loaded_data['sheet_2']['countrycode'] != 'DAV']

# and to check 'DAV' is now gone:
display(loaded_data['sheet_2'][loaded_data['sheet_2']['countrycode'] == 'DAV'])

,countrycode,country,currency_unit,year,rgdpna


In [153]:
for i in range(1, 6):
    sheet_i = loaded_data[f'sheet_{i}']
    sheet_i.dropna()
    loaded_data[f'sheet_{i}'] = sheet_i
    print(len(loaded_data[f'sheet_{i}']))

67
7730
7720
7720
7720
